A `generator` is a function or expression that returns a special type of **iterator** called **generator iterator**. It's special in the sense that they do not store their `values` in memory but *generate* their values when needed.

A generator function looks like any other function, but contains one or more **yield expressions** each `yield` will suspend code execution, saving the current execution state (*including all local variables and try-statements*). When the generator resumes, it picks up state from the suspension - unlike regular functions which reset with every call

### Constructing a generator

Generators are constructed much like other looping or recursive functions, but require a `yield` **expression**, which we will explore in depth a bit later. An example is a function that returns the *squares* from a given list of numbers. As currently written, all inputs must be processed before any values can be returned.

In [1]:
def squares(list_of_numbers):
    squares = []
    for number in list_of_numbers:
        squares.append(number ** 2)
    return squares

In [2]:
def squares_generator(list_of_numbers):
    for number in list_of_numbers:
        yield number ** 2

The rationale behind this is that you use a generator when you do not need to produce all the values at once. This saves memory and processing power, since only the value you are currently working on is calculated.

In [4]:
squared_numbers = squares_generator([1,2,3,4])

for square in squared_numbers:
    print(square)

1
4
9
16


Values within a generator can also be produced/accessed via the next() function. next() calls the __next__() method of a generator-iterator object, "advancing" or evaluating the code up to its yield expression, which then "yields" or returns a value:

In [11]:
squared_numbers = squares_generator([1,2])

In [142]:
type(squared_numbers)

generator

In [12]:
next(squared_numbers)

1

In [13]:
next(squared_numbers)

4

Generator-iterators are a special sub-set of iterators. Iterators are the mechanism/protocol that enables looping over iterables. Generator-iterators and the iterators returned by common Python iterables act very similarly, but there are some important differences to note:

* They are lazily evaluated; iteration is one-way and there is no "backing up" to a previous value.
* They are consumed by iterating over the returned values; there is no resetting or saving in memory.
* They are not sortable and cannot be reversed.
* They are not sequence types, and do not have indexes. You cannot reference a previous or future value using addition or subtraction and you cannot use bracket ([]) notation or slicing.
* They cannot be used with the len() function, as they have no length.
* They can be finite or infinite - be careful when collecting all values from an infinite generator-iterator!

### The yield expression

The **yield expression** is very similary to the `return` expression, *unlike* the `return` expression, `yield` gives up values to the caller at a *specific point*, suspending evaluation/return of any additionaly values until they are requested. When `yield` is evaluated, it pauses the execution of the enclosing function and returns any values of the function *at that point in time*. The function then *stays in scope* and when `__next__()` is called, execution resumes until `yield` is encountered again.

In [16]:
def infinite_sequence():
    current_number = 0
    while True:
        yield current_number
        current_number += 1

In [18]:
lets_try = infinite_sequence()
lets_try.__next__()
lets_try.__next__()

1

Whenever `__next__()` is called the `infinite_sequence` object will return the *previous number + 1*.

### Why create a generator?

Generators are useufl in a lot of applications. When working with a potentially large collection of values, you might not want to put all of them into memory. A generator can be used to work on larger data piece-by-piece, saving memory and improving performance. Generators are also very helpful when a process or calculation is *complex, expensive* or *infinite*.

## Generators - practical applications

Conda Airlines is the programming-world's biggest airline, with over 10,000 flights a day!

They are currently assigning all seats to passengers by hand; this will need to be automated.

They have asked you to create software to automate passenger seat assignments. They require your software to be memory efficient and performant.

Conda wants to generate seat letters for their airplanes. An airplane is made of rows of seats. Each row has 4 seats. The seats in each row are always named A, B, C, and D.
The first seat in the row is A, the second seat in the row is B, and so on. After reaching D, it should start again with A.

Implement a function generate_seat_letters(<number>) that accepts an int that holds how many seat letters to be generated. The function should then return an iterable of seat letters.

In [28]:
def generate_seat_letters(number):
    """
    :param number: int - total number of seat letters to be generated.
    :return: generator - generator that yields seat letters

    Seat letters are generated from A to D.
    After D it should start again with A.

    Example A, B, C, D

    """
    alphabet_index = 0
    for i in range(0, number):
        if alphabet_index == 0:
            alphabet_index += 1
            yield 'A'
        if alphabet_index == 1:
            alphabet_index += 1
            yield 'B'
        if alphabet_index == 2:
            alphabet_index += 1
            yield 'C'
        else:
            alphabet_index = 0
            yield 'D'




In [217]:
def generate_seat_letters(number):
    letters = ['A', 'B', 'C', 'D']
    for i in range(number):
        yield letters[i % 4]

In [34]:
next(letters)

'A'

Conda wants a system that can generate a given number of seats for their airplanes. Each airplane has 4 seats in each row. The rows are defined using numbers, starting from 1 and going up. The seats should be ordered, like: 1A, 1B, 1C, 1D, 2A, 2B, 2C, 2D, 3A, 3B, 3C, 3D, ...

Here is an example:

x	1	2
Row	5	21
Seat letter	A	D
Result	5A	21D

In [216]:
def generate_seats(number):
    """
    :param number: int - total number of seat letters to be generated.
    :return: generator - generator that yields seat letters

    Seat letters are generated from A to D.
    After D it should start again with A.

    Example A, B, C, D

    """
    letters = generate_seat_letters(number)
    for i in range(number):
        # row should only change after every fourth iteration so we integer divide
        row = i // 4 + 1
        if row >= 13:
            row += 1
        letter = next(letters)
        yield f"{row}{letter}"

In [218]:
seats = generate_seats(2)

In [221]:
next(seats)

StopIteration: 

In [186]:
def generate_seats(number):
    """Generate a series of identifiers for airline seats.

    :param number: int - total number of seats to be generated.
    :return: generator - generator that yields seat numbers.
    A seat number consists of the row number and the seat letter.
    There is no row 13.
    Each row has 4 seats.
    Seats should be sorted from low to high.
    Example: 3C, 3D, 4A, 4B

    """
    yield enumerate(generate_seat_letters(number))

In [187]:
seats = generate_seats(2)

In [190]:
next(seats)

StopIteration: 

Now that you have a function that generates seats, you can use it to assign seats to passengers.

Implement a function assign_seats(<passengers>) that accepts a list of passenger names. The function should then return a dictionary of passenger as key, and seat_number as value.

In [110]:
def assign_seats(passengers):
    seats = generate_seats(len(passengers))
    passenger_seatings = zip(passengers, seats)
    return dict(passenger_seatings)

In [ ]:
def assign_seats(passengers):
    return dict(zip(passengers, generate_seats(len(passengers))))

Conda Airlines would like to have a unique code for each ticket. Since they are a big airline, they have a lot of flights. This means that there are multiple flights with the same seat number. They want you to create a system that creates a unique ticket that is 12 characters long string code for identification.

This code begins with the assigned_seat followed by the flight_id. The rest of the code is appended by 0s.

Implement a function generate_codes(<seat_numbers>, <flight_id>) that accepts a list of seat_numbers and a string with the flight number. The function should then return a generator that yields a ticket_number.

In [137]:
def generate_codes(seat_numbers, flight_id):
    '''
    :param seat_numbers: list[str] - list of seat numbers
    :param flight_id: str - string containing the flight identifier
    :return:
    '''
    for seat in seat_numbers:
        num_zeros = (12-(len(f'{seat}{flight_id}')))
        zeros = '0'*num_zeros
        yield f'{seat}{flight_id}' + zeros


In [ ]:
def generate_codes(seat_numbers, flight_id):
    '''
    :param seat_numbers: list[str] - list of seat numbers
    :param flight_id: str - string containing the flight identifier
    :return:
    '''
    for seat in seat_numbers:
        yield f'{seat}{flight_id}'.ljust(12, '0')